# STREL Dataset Feature Analysis
Analyzes features where NHR_Stress is not NA

## Import Libraries

In [39]:
import pandas as pd
import numpy as np

## Load Data

In [40]:
df = pd.read_csv('../../data/raw/STREL_raw.csv')

## Filter Data (NHR_Stress NOT NA)

In [41]:
original_shape = df.shape[0]
df = df[df['NHR_Stress'].notna()].copy()
filtered_shape = df.shape[0]

print("="*80)
print("STREL DATASET FEATURE ANALYSIS (NHR_Stress NOT NA)")
print("="*80)
print(f"\nOriginal Dataset: {original_shape} rows")
print(f"Filtered Dataset: {filtered_shape} rows (removed {original_shape - filtered_shape} rows with NA NHR_Stress)")
print(f"Total Features: {df.shape[1]} columns")

STREL DATASET FEATURE ANALYSIS (NHR_Stress NOT NA)

Original Dataset: 18425 rows
Filtered Dataset: 11176 rows (removed 7249 rows with NA NHR_Stress)
Total Features: 43 columns


## Separate Numeric and Categorical Columns

In [42]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

columns_to_exclude = ['Time']
numeric_cols = [col for col in numeric_cols if col not in columns_to_exclude]
categorical_cols = [col for col in categorical_cols if col not in columns_to_exclude]

print(f"Numeric Features: {len(numeric_cols)}")
print(f"Categorical Features: {len(categorical_cols)}")
print(f"Excluded from analysis: {', '.join(columns_to_exclude)}")

Numeric Features: 31
Categorical Features: 11
Excluded from analysis: Time


## Numeric Features Statistics

In [43]:
print("="*80)
print("NUMERIC FEATURES STATISTICS")
print("="*80)

numeric_stats = []

for col in numeric_cols:
    stats = {
        'Feature': col,
        'Count': df[col].count(),
        'Missing': df[col].isna().sum(),
        'Missing %': f"{(df[col].isna().sum() / len(df)) * 100:.2f}%",
        'Min': df[col].min(),
        'Max': df[col].max(),
        'Mean': df[col].mean(),
        'Median': df[col].median(),
        'Std': df[col].std()
    }
    numeric_stats.append(stats)

numeric_df = pd.DataFrame(numeric_stats)

print(f"\n{'Feature':<20} {'Count':<8} {'Missing':<10} {'Min':<12} {'Max':<12} {'Mean':<12} {'Median':<12} {'Std':<12}")
print("-"*120)

for _, row in numeric_df.iterrows():
    print(f"{row['Feature']:<20} {row['Count']:<8.0f} {row['Missing %']:<10} "
          f"{row['Min']:<12.2f} {row['Max']:<12.2f} {row['Mean']:<12.2f} "
          f"{row['Median']:<12.2f} {row['Std']:<12.2f}")

NUMERIC FEATURES STATISTICS

Feature              Count    Missing    Min          Max          Mean         Median       Std         
------------------------------------------------------------------------------------------------------------------------
Age                  11176    0.00%      27.00        58.00        41.34        42.00        7.34        
BMI                  11176    0.00%      20.80        38.30        25.80        24.40        3.94        
HR                   11176    0.00%      41.82        153.06       78.25        77.62        13.10       
RR                   10955    1.98%      0.40         1.44         0.80         0.79         0.14        
Cadence              11176    0.00%      0.00         61.25        4.16         0.00         7.03        
Speed                11176    0.00%      -0.01        104.37       4.26         1.31         11.40       
PNSindex             11176    0.00%      -4.22        4.25         -1.06        -1.14        1.03        
SN

## Categorical Features Statistics

In [44]:
print("="*80)
print("CATEGORICAL FEATURES STATISTICS")
print("="*80)

for col in categorical_cols:
    print(f"\n{'─'*80}")
    print(f"Feature: {col}")
    print(f"{'─'*80}")
    
    total_count = df[col].count()
    missing_count = df[col].isna().sum()
    missing_pct = (missing_count / len(df)) * 100
    unique_count = df[col].nunique()
    
    print(f"Total Values: {total_count}")
    print(f"Missing: {missing_count} ({missing_pct:.2f}%)")
    print(f"Unique Categories: {unique_count}")
    
    value_counts = df[col].value_counts()
    value_percentages = df[col].value_counts(normalize=True) * 100
    
    print(f"\nCategory Distribution:")
    print(f"{'Category':<30} {'Count':<10} {'Percentage':<12}")
    print("-"*52)
    
    for category, count in value_counts.items():
        pct = value_percentages[category]
        print(f"{str(category):<30} {count:<10} {pct:>6.2f}%")

CATEGORICAL FEATURES STATISTICS

────────────────────────────────────────────────────────────────────────────────
Feature: Participant
────────────────────────────────────────────────────────────────────────────────
Total Values: 11176
Missing: 0 (0.00%)
Unique Categories: 24

Category Distribution:
Category                       Count      Percentage  
----------------------------------------------------
T004                           694          6.21%
T025                           642          5.74%
T023                           594          5.31%
T005                           571          5.11%
T015                           563          5.04%
T016                           558          4.99%
T021                           545          4.88%
T002                           539          4.82%
T020                           536          4.80%
T014                           534          4.78%
T007                           532          4.76%
T019                           463       

## Summary Report

In [45]:
print("="*80)
print("SUMMARY REPORT")
print("="*80)

total_missing = df.isna().sum().sum()
total_cells = df.shape[0] * df.shape[1]
missing_pct = (total_missing / total_cells) * 100

print(f"\nOverall Missing Data: {total_missing:,} cells ({missing_pct:.2f}%)")

missing_by_feature = df.isna().sum().sort_values(ascending=False)
missing_by_feature = missing_by_feature[missing_by_feature > 0]

if len(missing_by_feature) > 0:
    print(f"\nFeatures with Missing Data:")
    print(f"{'Feature':<30} {'Missing Count':<15} {'Percentage'}")
    print("-"*60)
    for feature, count in missing_by_feature.head(10).items():
        pct = (count / len(df)) * 100
        print(f"{feature:<30} {count:<15} {pct:>6.2f}%")
else:
    print("\n✓ No missing data found!")

SUMMARY REPORT

Overall Missing Data: 221 cells (0.05%)

Features with Missing Data:
Feature                        Missing Count   Percentage
------------------------------------------------------------
RR                             221               1.98%


## Save Results

In [46]:
numeric_df.to_csv('../../results/eda/numeric_features_stats.csv', index=False)
print("Analysis complete! Results saved to results/eda/numeric_features_stats.csv")

Analysis complete! Results saved to results/eda/numeric_features_stats.csv
